In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel,Field
import operator

In [2]:
load_dotenv()  # Load environment variables from .env file   

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    max_output_tokens=512,
    thinking_budget=0,
)

In [4]:
from pydantic import BaseModel, Field

class EvaluationSchema(BaseModel):
  feedback: str = Field(..., description="Feedback on the answer")
  score: int = Field(..., description="Score for the answer out of 10", ge=0, le=10)

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(
  model="gemini-3.5-flash",
  temperature=0,
  max_output_tokens=512,
  thinking_budget=0,
)

structured_model = model.with_structured_output(EvaluationSchema)

In [6]:
essay="""
The Industrial Revolution was a period of significant technological, socioeconomic, and cultural change that began in the late 18th century and continued into the 19th century. It marked a shift from agrarian economies to industrialized and urban societies, primarily in Europe and North America. The revolution was characterized by the development of new machinery, the rise of factories, and the mass production of goods.
"""

In [7]:
prompt = f"""
You are an expert in evaluating essays. Please read the following essay and provide feedback and a score
"""
result = structured_model.invoke(
  f"""
You are an expert in evaluating essays. Please read the following essay and provide feedback and a score.

Essay:
{essay}
"""
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [8]:
class CSSState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int],operator.add]
    avg_score: float

In [9]:
def evaluate_language(state: CSSState):
    prompt = f"""Evaluate the following essay for language quality — grammar, vocabulary, and sentence structure. Provide feedback and a score out of 10.

Essay:
{state['essay']}
"""
    result = structured_model.invoke(prompt)
    return {'language_feedback': result.feedback, 'individual_scores': [result.score]}

In [10]:
def evaluate_analysis(state: CSSState):
    prompt = f"""Evaluate the following essay for depth of analysis — how well it explores causes, effects, and connects ideas. Provide feedback and a score out of 10.

Essay:
{state['essay']}
"""
    result = structured_model.invoke(prompt)
    return {'analysis_feedback': result.feedback, 'individual_scores': [result.score]}

In [11]:
def evaluate_thought(state: CSSState):
    prompt = f"""Evaluate the following essay for clarity of thought — how clearly and logically the ideas are organized and expressed. Provide feedback and a score out of 10.

Essay:
{state['essay']}
"""
    result = structured_model.invoke(prompt)
    return {'clarity_feedback': result.feedback, 'individual_scores': [result.score]}

In [12]:
def final_evaluation(state: CSSState):
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    prompt = f"""Combine the following feedback on an essay into a short overall summary (2-3 sentences).

Language feedback: {state['language_feedback']}
Analysis feedback: {state['analysis_feedback']}
Clarity feedback: {state['clarity_feedback']}
"""
    overall_feedback = model.invoke(prompt).content

    return {'avg_score': avg_score, 'overall_feedback': overall_feedback}

In [13]:
graph=StateGraph(CSSState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

# fan-out: all three evaluators run in parallel from START
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

# fan-in: final_evaluation runs once all three finish
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [14]:
initial_state = {'essay': essay}
final_state = workflow.invoke(initial_state)

print("Language feedback:", final_state['language_feedback'])
print("Analysis feedback:", final_state['analysis_feedback'])
print("Clarity feedback:", final_state['clarity_feedback'])
print("Individual scores:", final_state['individual_scores'])
print("Average score:", final_state['avg_score'])
print("Overall feedback:", final_state['overall_feedback'])

Language feedback: The essay excerpt demonstrates exceptional language quality. The grammar is flawless, the vocabulary is precise and academic (e.g., 'agrarian economies', 'socioeconomic', 'industrialized'), and the sentence structure is varied and flows logically. It provides a highly clear and sophisticated introduction to the topic.
Analysis feedback: The essay provides a clear and accurate definition of the Industrial Revolution, identifying key characteristics such as the shift from agrarian to industrial societies, the rise of factories, and mass production. However, it lacks depth of analysis. It merely states that these changes occurred without exploring the underlying causes (such as why it began in Britain, the role of capitalism, or specific technological innovations like the steam engine) or the long-term effects (such as urbanization, labor movements, environmental impacts, or social stratification). To improve, the essay needs to connect these ideas and analyze the 'why'